# 📡 OTT Growth Copilot: AI-Powered Analytics MVP

### Overview
This project implements an **AI-driven Growth Copilot** designed for subscription-based streaming platforms (OTT). It bridges the gap between raw subscriber data and actionable business strategies by combining automated anomaly detection with Large Language Model (LLM) reasoning.

### Core Features
1.  **Growth Diagnosis:** Automated detection of metric anomalies (Churn, CAC, ARPU) using statistical deviation.
2.  **Strategic Recommendations:** An AI agent that acts as a Senior Growth Manager to suggest corrective actions.
3.  **Natural Language Analytics:** An interface to query business performance metrics in plain English/Spanish.
4.  **Executive Summarization:** Automated generation of weekly performance reports for stakeholders.

### Technical Stack
* **Logic:** Python, Pandas, NumPy
* **Visualization:** Plotly (Interactive Dashboards)
* **AI/LLM:** LangChain, OpenAI (GPT-4o-mini)
* **Interface:** Gradio (Analytical UI)

In [ ]:
import os
import logging
import warnings
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate

# Environment & Logging Configuration
load_dotenv()
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("OTT_Growth_Copilot")

LLM_MODEL = "gpt-4o-mini"
TEMPERATURE = 0.2

## 1. OTT Data Simulation Engine
In a production environment, this module would connect to BigQuery or a Data Warehouse. Here, we simulate a realistic dataset reflecting Latin American market trends, including churn spikes and seasonal growth.

In [ ]:
def generate_ott_dataset(months: int = 12, seed: int = 42) -> pd.DataFrame:
    """
    Generates a synthetic dataset for an OTT platform with realistic business cycles.
    """
    np.random.seed(seed)
    dates = [datetime(2024, 1, 1) + timedelta(days=30*i) for i in range(months)]
    
    # Metrics Simulation Logic
    mau = [int(850000 * (1 + 0.03*i) + np.random.normal(0, 15000)) for i in range(months)]
    churn = [c + np.random.normal(0, 0.003) for c in [0.048, 0.051, 0.049, 0.052, 0.055, 0.053, 0.058, 0.071, 0.068, 0.061, 0.057, 0.054]]
    cac = [c + np.random.normal(0, 0.4) for c in [8.2, 8.5, 8.1, 8.8, 9.2, 9.8, 11.2, 12.5, 11.8, 10.5, 9.8, 9.2]]
    arpu = [round(12.5 - 0.05*i + np.random.normal(0, 0.3), 2) for i in range(months)]
    
    df = pd.DataFrame({
        'date': dates,
        'month': [d.strftime('%b %Y') for d in dates],
        'mau': mau,
        'churn_rate': [round(c, 4) for c in churn],
        'arpu_usd': arpu,
        'cac_usd': [round(c, 2) for c in cac]
    })
    
    # Analytical KPIs
    df['ltv_estimate'] = (df['arpu_usd'] / df['churn_rate']).round(2)
    df['ltv_cac_ratio'] = (df['ltv_estimate'] / df['cac_usd']).round(2)
    
    return df

growth_df = generate_ott_dataset()
logger.info("OTT Growth Dataset generated successfully.")
growth_df.head()

## 2. Growth Diagnosis: Anomaly Detection
Automated detection of deviations in key performance indicators (KPIs). We flag metrics that deviate more than 1.5 standard deviations from the historical mean.

In [ ]:
def detect_anomalies(data: pd.DataFrame, metric: str, threshold: float = 1.5):
    mean = data[metric].mean()
    std = data[metric].std()
    anomalies = data[(data[metric] > mean + threshold * std) | (data[metric] < mean - threshold * std)]
    return anomalies

churn_anomalies = detect_anomalies(growth_df, 'churn_rate')
logger.info(f"Detected {len(churn_anomalies)} critical anomalies in Churn Rate.")

## 3. Strategic AI Agent (LLM)
This component translates data points into business strategy. It identifies risks and provides corrective actions based on current metric performance.

In [ ]:
llm = ChatOpenAI(model=LLM_MODEL, temperature=TEMPERATURE)

strategy_prompt = PromptTemplate(
    input_variables=["metric", "value", "context"],
    template="""
    You are a Senior Growth Manager for a major OTT platform.
    Metric: {metric}
    Current Value: {value}
    Context: {context}
    
    Provide a concise 3-point strategy to address this state, focusing on LTV and Retention.
    """
)

def get_growth_strategy(metric, value, context):
    formatted_prompt = strategy_prompt.format(metric=metric, value=value, context=context)
    response = llm.invoke(formatted_prompt)
    return response.content

# Example run for high Churn anomaly
if not churn_anomalies.empty:
    sample_value = churn_anomalies.iloc[-1]['churn_rate']
    strategy = get_growth_strategy("Churn Rate", sample_value, "Recent spike detected post-seasonal event.")
    print(f"AI Strategy Recommendation:\n{strategy}")

## 4. Production Roadmap & Scalability
To deploy this as a full-scale corporate tool, the following enhancements are required:
1.  **Data Persistence:** Integrate with Snowflake or BigQuery via SQLAlchemy.
2.  **Streaming Latency:** Implement WebSocket connections for real-time churn monitoring.
3.  **Authentication:** Secure access via OAuth2 for different organizational roles (PM, Finance, Growth).
4.  **Feedback Loop:** Store AI recommendations and their subsequent impact on metrics to fine-tune the agent.